In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import kendalltau
import os
import warnings
warnings.filterwarnings('ignore')

# Load data
file_path = r"C:\Users\jdelhoyo\PhD\Study cases\Genissiat\Arve-Valserine\Isotopes\GitHub Isotopes\Data\OnlyDataIsotopes.xlsx"
df = pd.read_excel(file_path)

print(f"Data loaded: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nFirst rows:\n{df.head()}")

In [ ]:
# Explore data categories
print("Unique species:")
print(df['Sp'].unique())
print(f"\nSpecies counts:\n{df['Sp'].value_counts()}")

print("\n" + "="*50)
print("Unique conditions:")
print(df['Cond'].unique())
print(f"\nCondition counts:\n{df['Cond'].value_counts()}")

print("\n" + "="*50)
print("Species x Condition crosstab:")
print(pd.crosstab(df['Sp'], df['Cond']))

In [ ]:
# Create 6 groups: Sp (Fraxinus, Alnus) + Living + Condition (Float, Out, Sub)
grupos = {
    'Fraxinus_Living_Float': df[(df['Sp'] == 'Fraxinus') & (df['Cond'].isin(['Living', 'Float']))],
    'Fraxinus_Living_Out': df[(df['Sp'] == 'Fraxinus') & (df['Cond'].isin(['Living', 'Out']))],
    'Fraxinus_Living_Sub': df[(df['Sp'] == 'Fraxinus') & (df['Cond'].isin(['Living', 'Sub']))],
    'Alnus_Living_Float': df[(df['Sp'] == 'Alnus') & (df['Cond'].isin(['Living', 'Float']))],
    'Alnus_Living_Out': df[(df['Sp'] == 'Alnus') & (df['Cond'].isin(['Living', 'Out']))],
    'Alnus_Living_Sub': df[(df['Sp'] == 'Alnus') & (df['Cond'].isin(['Living', 'Sub']))]
}

print("\n" + "="*70)
print("6 GROUPS CREATED")
print("="*70)

for nombre, grupo_df in grupos.items():
    print(f"\n{nombre}: {len(grupo_df)} samples")
    print(f"  D: {grupo_df['D'].mean():.2f} ± {grupo_df['D'].std():.2f}")
    print(f"  O: {grupo_df['O'].mean():.2f} ± {grupo_df['O'].std():.2f}")

In [ ]:
# Mann-Kendall test for each group
print("\n" + "="*100)
print("MANN-KENDALL TEST: ONE TEST PER GROUP")
print("="*100)

mk_results = {}

for nombre_grupo, grupo_df in grupos.items():
    print(f"\n{nombre_grupo}:")
    
    # Calculate mean D and O for each Order
    time_series = grupo_df.groupby('Order')[['D', 'O']].mean().reset_index().sort_values('Order')
    time_points = time_series['Order'].values
    mean_d = time_series['D'].values
    mean_o = time_series['O'].values
    n_timepoints = len(time_series)
    
    print(f"  Time points: {n_timepoints} Orders")
    print(f"  Orders: {sorted([int(x) for x in time_points])}")
    
    # Apply Mann-Kendall test
    time_indices = np.arange(n_timepoints)
    tau_d, p_d = kendalltau(time_indices, mean_d)
    tau_o, p_o = kendalltau(time_indices, mean_o)
    
    trend_d = "Increasing" if tau_d > 0 else "Decreasing"
    sig_d = "***" if p_d < 0.05 else "NS"
    trend_o = "Increasing" if tau_o > 0 else "Decreasing"
    sig_o = "***" if p_o < 0.05 else "NS"
    
    print(f"  D: τ={tau_d:.4f}, p={p_d:.4f} ({trend_d}, {sig_d})")
    print(f"  O: τ={tau_o:.4f}, p={p_o:.4f} ({trend_o}, {sig_o})")
    
    mk_results[nombre_grupo] = {
        'n_timepoints': n_timepoints,
        'D_tau': tau_d, 'D_pval': p_d, 'D_trend': trend_d,
        'O_tau': tau_o, 'O_pval': p_o, 'O_trend': trend_o
    }

In [ ]:
# Descriptive statistics: mean and std by Order for each group
print("\n" + "="*100)
print("DESCRIPTIVE STATISTICS: MEAN AND STD BY ORDER")
print("="*100)

descriptive_stats = {}

for nombre_grupo, grupo_df in grupos.items():
    print(f"\n{nombre_grupo}:")
    
    stats_by_order = grupo_df.groupby('Order')[['D', 'O']].agg(['mean', 'std', 'count']).reset_index().sort_values('Order')
    stats_by_order.columns = ['Order', 'D_mean', 'D_std', 'D_count', 'O_mean', 'O_std', 'O_count']
    
    print(f"{'Order':<8} {'N':<6} {'D_mean':<14} {'D_std':<14} {'O_mean':<14} {'O_std':<14}")
    for idx, row in stats_by_order.iterrows():
        print(f"{int(row['Order']):<8} {int(row['D_count']):<6} {row['D_mean']:<14.4f} {row['D_std']:<14.4f} {row['O_mean']:<14.4f} {row['O_std']:<14.4f}")
    
    descriptive_stats[nombre_grupo] = stats_by_order

In [ ]:
# Group-level extremes: find max/min for each group and calculate deviations
print("\n" + "="*100)
print("GROUP-LEVEL EXTREMES ANALYSIS")
print("="*100)

group_extremes = {}

for nombre_grupo, grupo_df in grupos.items():
    print(f"\n{nombre_grupo}:")
    
    # Get time series for this group
    time_series = grupo_df.groupby('Order')[['D', 'O']].mean().reset_index().sort_values('Order')
    mean_d_values = time_series['D'].values
    mean_o_values = time_series['O'].values
    orders = time_series['Order'].values
    
    # Get baseline (Order 0)
    baseline_d = time_series[time_series['Order'] == 0]['D'].values[0]
    baseline_o = time_series[time_series['Order'] == 0]['O'].values[0]
    
    print(f"  Baseline (Order 0): D={baseline_d:.4f}, O={baseline_o:.4f}")
    
    # Find extremes
    max_d_idx = np.argmax(mean_d_values)
    min_d_idx = np.argmin(mean_d_values)
    max_o_idx = np.argmax(mean_o_values)
    min_o_idx = np.argmin(mean_o_values)
    
    max_d_val = mean_d_values[max_d_idx]
    max_d_ord = int(orders[max_d_idx])
    max_d_dev = max_d_val - baseline_d
    
    min_d_val = mean_d_values[min_d_idx]
    min_d_ord = int(orders[min_d_idx])
    min_d_dev = min_d_val - baseline_d
    
    max_o_val = mean_o_values[max_o_idx]
    max_o_ord = int(orders[max_o_idx])
    max_o_dev = max_o_val - baseline_o
    
    min_o_val = mean_o_values[min_o_idx]
    min_o_ord = int(orders[min_o_idx])
    min_o_dev = min_o_val - baseline_o
    
    print(f"  D MAX: Order {max_d_ord}, Value={max_d_val:.4f}, Deviation={max_d_dev:+.4f}")
    print(f"  D MIN: Order {min_d_ord}, Value={min_d_val:.4f}, Deviation={min_d_dev:+.4f}")
    print(f"  O MAX: Order {max_o_ord}, Value={max_o_val:.4f}, Deviation={max_o_dev:+.4f}")
    print(f"  O MIN: Order {min_o_ord}, Value={min_o_val:.4f}, Deviation={min_o_dev:+.4f}")
    
    group_extremes[nombre_grupo] = {
        'Baseline_D': baseline_d, 'Baseline_O': baseline_o,
        'Max_D_val': max_d_val, 'Max_D_ord': max_d_ord, 'Max_D_dev': max_d_dev,
        'Min_D_val': min_d_val, 'Min_D_ord': min_d_ord, 'Min_D_dev': min_d_dev,
        'Max_O_val': max_o_val, 'Max_O_ord': max_o_ord, 'Max_O_dev': max_o_dev,
        'Min_O_val': min_o_val, 'Min_O_ord': min_o_ord, 'Min_O_dev': min_o_dev
    }

In [ ]:
# Export all results to Excel
output_dir = r"C:\Users\jdelhoyo\PhD\Study cases\Genissiat\Arve-Valserine\Isotopes\GitHub Isotopes\Results"
os.makedirs(output_dir, exist_ok=True)

print("\n" + "="*100)
print("EXPORTING RESULTS TO EXCEL")
print("="*100)

# 1. Mann-Kendall results
mk_data = []
for nombre, result in mk_results.items():
    mk_data.append({
        'Group': nombre, 'Variable': 'D', 'N': result['n_timepoints'],
        'Kendall_tau': result['D_tau'], 'p_value': result['D_pval'], 'Trend': result['D_trend']
    })
    mk_data.append({
        'Group': nombre, 'Variable': 'O', 'N': result['n_timepoints'],
        'Kendall_tau': result['O_tau'], 'p_value': result['O_pval'], 'Trend': result['O_trend']
    })
mk_df = pd.DataFrame(mk_data)

# 2. Group extremes
extreme_data = []
for nombre, result in group_extremes.items():
    extreme_data.append({
        'Group': nombre, 'Variable': 'D', 'Type': 'MAX',
        'Value': result['Max_D_val'], 'Order': result['Max_D_ord'], 'Deviation': result['Max_D_dev']
    })
    extreme_data.append({
        'Group': nombre, 'Variable': 'D', 'Type': 'MIN',
        'Value': result['Min_D_val'], 'Order': result['Min_D_ord'], 'Deviation': result['Min_D_dev']
    })
    extreme_data.append({
        'Group': nombre, 'Variable': 'O', 'Type': 'MAX',
        'Value': result['Max_O_val'], 'Order': result['Max_O_ord'], 'Deviation': result['Max_O_dev']
    })
    extreme_data.append({
        'Group': nombre, 'Variable': 'O', 'Type': 'MIN',
        'Value': result['Min_O_val'], 'Order': result['Min_O_ord'], 'Deviation': result['Min_O_dev']
    })
extreme_df = pd.DataFrame(extreme_data)

# Write to Excel with multiple sheets
with pd.ExcelWriter(os.path.join(output_dir, 'Isotope_Analysis_Complete.xlsx'), engine='openpyxl') as writer:
    mk_df.to_excel(writer, sheet_name='Mann-Kendall', index=False)
    extreme_df.to_excel(writer, sheet_name='Group-Extremes', index=False)
    for nombre, stats_df in descriptive_stats.items():
        sheet_name = nombre[:31]  # Excel limit
        stats_df.to_excel(writer, sheet_name=sheet_name, index=False)

print(f"\n✓ File saved: {os.path.join(output_dir, 'Isotope_Analysis_Complete.xlsx')}")
print("\nSheets created:")
print("  1. Mann-Kendall: Test results for each group")
print("  2. Group-Extremes: Max/min values and deviations for each group")
print(f"  3-8. Group statistics: Descriptive stats by Order for each group")